In [1]:
import ROOT as r

In [2]:
prepath = "/eos/user/a/aiuliano/public/sims_FairShip/GenieEvents_SHIP/GenieEvents_2026_03/2026_03_16_1year_allflavours/"
chaingst = r.TChain("gst")
chaingst.Add(prepath+"nu_1year_fluxhanae34.gst.root")
chaingst.AddFile("/eos/experiment/sndlhc/users/aiulian/Neutrino_MC_merged/sndlhc_13TeV_down_volTarget_100fb-1_SNDG18_02a_01_000/sndlhc_+volTarget_all_SNDG18_02a_01_000.0.gst.root")

chainrootracker = r.TChain("gRooTracker")
chainrootracker.Add(prepath+"nu_1year_fluxhanae34.rootracker.root")
chainrootracker.AddFile("/eos/experiment/sndlhc/users/aiulian/Neutrino_MC_merged/sndlhc_13TeV_down_volTarget_100fb-1_SNDG18_02a_01_000/sndlhc_+volTarget_all_SNDG18_02a_01_000.0.rootracker.root")
#gstfile = r.TFile.Open(prepath+"nu_1year_fluxhanae34.gst.root")
#rootrackerfile = r.TFile.Open(prepath+"nu_1year_fluxhanae34.rootracker.root")

#gstfile = r.TFile.Open("/eos/experiment/sndlhc/users/aiulian/Neutrino_MC_merged/sndlhc_13TeV_down_volTarget_100fb-1_SNDG18_02a_01_000/sndlhc_+volTarget_all_SNDG18_02a_01_000.0.gst.root")
#rootrackerfile = r.TFile.Open("/eos/experiment/sndlhc/users/aiulian/Neutrino_MC_merged/sndlhc_13TeV_down_volTarget_100fb-1_SNDG18_02a_01_000/sndlhc_+volTarget_all_SNDG18_02a_01_000.0.rootracker.root")

#gsttree = gstfile.Get("gst")
#rootrackertree = rootrackerfile.Get("gRooTracker")

#gsttree.AddFriend(rootrackertree)
chaingst.AddFriend(chainrootracker)

In [3]:
df = r.RDataFrame(chaingst);

In [4]:
df2 = df.Define("xsecN_overE","gRooTracker.EvtXSec/(Ev*184)")

In [5]:
#filter per neutrino type
df_numu = df2.Filter("neu==14");
#target must be tungsten
df_numu_W = df_numu.Filter("tgt==1000741840");

In [6]:
nbinsE = 200;
minE = 0.;
maxE = 2000.;

nbinsxsec = 1000;
minxsec = 0.;
maxxsec = 1.;

In [7]:
def selectchannel(cc,inttype,hitnuc,charm,sea = "None",hitqrk = "None", resid = "None"):
 dfint = df_numu_W.Filter(cc+"&&"+inttype+"&&"+hitnuc+"&&"+charm)
 if (inttype=="dis"):
     dfintqrk = dfint.Filter(sea+"&&"+hitqrk)
     return dfintqrk
 if (inttype=="res"):
     dfintqrk = dfint.Filter(resid)
     return dfintqrk
 
 else: 
     return dfint
    

In [8]:
charge = ["cc"]
charms = ["charm","!charm"]
inttypes = ["qel","mec","coh","imd","nuel"]


hitnuclei = ["hitnuc==2212","hitnuc==2112"]

In [9]:
profiles = []
for cc in charge:
    for inttype in inttypes:
        for hitnuc in hitnuclei:
            for charm in charms:
                selectdf = selectchannel(cc, inttype, hitnuc,charm);
                profiles.append(selectdf.Profile1D(("prof_"+cc+"_"+inttype+"_"+hitnuc+"_"+charm,"Int Profile",nbinsE,minE,maxE,minxsec,maxxsec),"Ev","xsecN_overE"))
            
    

In [10]:
#ccres need to loop over all resids
resonances = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17]

for resid in resonances:
    for hitnuc in hitnuclei:
        for charm in charms:
            selectdf = selectchannel("cc", "res", hitnuc,charm,resid=f"resid=={resid}")
            profiles.append(selectdf.Profile1D(("prof_"+cc+"_"+inttype+"_"+hitnuc+"_"+charm+f"_resid_{resid}","Int Profile",nbinsE,minE,maxE,minxsec,maxxsec),"Ev","xsecN_overE"))


In [11]:
#profiles = []
#ccdis need to loop over all quarks
quarks = [-6,-5,-4,-3,-2,-1,1,2,3,4,5,6]
sea_valence = ["!sea","sea"]
charge = ["cc"]
hitnuclei = ["hitnuc==2212","hitnuc==2112"]

for cc in charge:
    for charm in charms:
        for hitnuc in hitnuclei:
            for issea in sea_valence:
                for quark in quarks:
                    selectdf = selectchannel(cc,"dis",hitnuc,charm,issea,f"hitqrk=={quark}")
                    profiles.append(selectdf.Profile1D(("prof_"+cc+"_dis_"+hitnuc+"_"+charm+"_"+issea+f"_quark_{quark}","Int Profile",nbinsE,minE,maxE,minxsec,maxxsec),"Ev","xsecN_overE"));

In [12]:
h_sum = r.TH1D("h_sum","total cross section",nbinsE,minE,maxE)
for profile in profiles:
    h_sum.Add(profile.ProjectionX("h_"+profile.GetName()))

In [13]:
c0 = r.TCanvas()
h_sum.Draw("histo")
c0.Draw()

In [14]:
for prof in profiles:
    if (prof.GetEntries() > 0):
        print(prof.GetName(),prof.GetEntries())

prof_cc_qel_hitnuc==2212_charm 68.0
prof_cc_qel_hitnuc==2112_charm 277.0
prof_cc_qel_hitnuc==2112_!charm 7467.0
prof_cc_nuel_hitnuc==2212_!charm_resid_0 5128.0
prof_cc_nuel_hitnuc==2112_!charm_resid_0 2572.0
prof_cc_nuel_hitnuc==2112_!charm_resid_1 1760.0
prof_cc_nuel_hitnuc==2112_!charm_resid_2 1640.0
prof_cc_nuel_hitnuc==2112_!charm_resid_3 61.0
prof_cc_nuel_hitnuc==2112_!charm_resid_4 104.0
prof_cc_nuel_hitnuc==2112_!charm_resid_5 180.0
prof_cc_nuel_hitnuc==2212_!charm_resid_6 84.0
prof_cc_nuel_hitnuc==2112_!charm_resid_6 54.0
prof_cc_nuel_hitnuc==2212_!charm_resid_7 259.0
prof_cc_nuel_hitnuc==2112_!charm_resid_7 123.0
prof_cc_nuel_hitnuc==2112_!charm_resid_8 1360.0
prof_cc_nuel_hitnuc==2212_!charm_resid_9 604.0
prof_cc_nuel_hitnuc==2112_!charm_resid_9 303.0
prof_cc_nuel_hitnuc==2112_!charm_resid_10 672.0
prof_cc_nuel_hitnuc==2112_!charm_resid_11 793.0
prof_cc_nuel_hitnuc==2212_!charm_resid_12 8.0
prof_cc_nuel_hitnuc==2112_!charm_resid_12 7.0
prof_cc_nuel_hitnuc==2212_!charm_resid_1

In [15]:
!pwd

/eos/home-i01/a/aiuliano/SWAN_projects/macros-ship/nu_ship_simulations
